# M01 — What is represented, and in which direction?

<!-- paper-first -->
### Research question

**Reading:** [PM01](../../curriculum/papers/modeling.md#pm01). Review the assigned figure or result before starting the lesson.

**Question:** Does this model predict brain measurements from a stimulus or predict a target from brain measurements?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

A brain image is not yet a learning problem. We must choose a unit of observation, a numerical representation, a target, and the population to which the result should apply. A row might represent one participant, a stimulus event, or one time sample. Those choices change the available independent evidence. In this lab a row is one independent invented stimulus presentation; in a continuous fMRI run neighboring rows would be dependent, so the later temporal-splitting lesson would apply.

Encoding predicts measurements from explanatory features: stimulus description → expected response. Decoding predicts a stimulus, behavior, or outcome from measurements: response pattern → target. Both directions can use regression. Successful decoding does not establish that the brain implements the fitted decoder or that a predictive voxel causes the outcome. Successful encoding supports an association between the chosen features and responses on the evaluation data; competing feature descriptions can still predict similarly.

We construct three stimulus features and eight measured channels. A known matrix maps stimulus features into channels, then noise is added. This makes the distinction inspectable: the encoding coefficients have three rows and eight columns, while the decoding coefficients map eight measurements into three outputs. The reverse problem is not simply the transpose of the forward problem because noise, correlations, and regularization matter. A real feature map might represent sound amplitude, word semantics, stimulus categories, or task variables convolved with an HRF.

Averaging channels produces a region summary. It may improve precision when channels share a signal, but opposite tuning can cancel. We deliberately place positive and negative copies of one signal into neighboring channels. Their average has little relationship to that stimulus despite each channel carrying useful information. A representation therefore answers a question; it is not an automatically neutral compression. Preserve the channel order and the definition of every feature, since a model receives positions in a matrix rather than meaningful column names.

The same held-out rows are used for both directions in this synthetic exercise. For real within-person decoding, hold out independent runs or blocks; for claims about new people, hold out people. Delay structure, stimulus repetitions, and scanner processing can link otherwise distinct rows. Ask the AI to name these dependencies before letting it propose a split. No model score can repair a split that tests the wrong claim.

## Transformation contract

Stimulus matrix S `(240,3)` and responses Y `(240,8)` → fitted encoding and decoding maps → held-out predictions. Averaging two channels changes `(240,2)` to `(240,)` and loses their difference; identity and timing metadata must travel separately.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
rng = np.random.default_rng(401)
S = rng.normal(size=(240,3))
W = rng.normal(size=(3,8)); W[:,0] = [1,0,0]; W[:,1] = [-1,0,0]
Y = S @ W + rng.normal(0,.12,(240,8))
tr, te = np.arange(160), np.arange(160,240)
encoding = Ridge(alpha=1).fit(S[tr],Y[tr])
decoding = Ridge(alpha=1).fit(Y[tr],S[tr])
enc_r2 = r2_score(Y[te],encoding.predict(S[te]))
dec_r2 = r2_score(S[te],decoding.predict(Y[te]))
print('Encoding / decoding held-out R²:',enc_r2,dec_r2)
assert encoding.coef_.shape == (8,3) and decoding.coef_.shape == (3,8)
assert enc_r2 > .8 and dec_r2 > .8


Encoding / decoding held-out R²: 0.9817933292612452 0.9954356798612215


**Predict the failure:** what happens when channels with opposing responses are averaged?

In [2]:
region = Y[:,:2].mean(axis=1)
individual = np.corrcoef(S[:,0],Y[:,0])[0,1]
averaged = np.corrcoef(S[:,0],region)[0,1]
print('Stimulus correlation: first channel / averaged region',individual,averaged)
assert abs(individual) > .95 and abs(averaged) < .2
shuffled = rng.permutation(Y[te])
print('Encoding R² after scrambling row correspondence:',r2_score(shuffled,encoding.predict(S[te])))
assert r2_score(shuffled,encoding.predict(S[te])) < 0


Stimulus correlation: first channel / averaged region 0.993649961139653 -0.05892500469494497
Encoding R² after scrambling row correspondence: -0.3718094016972846


## Deliberate failure and repair

The code demonstrates two different failures: averaging cancels opposing tuning, and row permutation destroys the stimulus–response correspondence. Neither failure changes the number of observations. Repair the first by selecting a representation that preserves the relevant contrast; repair the second using genuine event/run identifiers, not by searching for the permutation with the best score.

## Your investigation

Draw encoding and decoding arrows with array dimensions. Propose one stimulus feature for a movie experiment and explain its timing. Compare a voxel pattern with an ROI mean; name information each retains and discards. Write a validation claim about either new runs or new participants and justify the corresponding split.

## Transfer to real neuroimaging

The NMA encoding tutorial uses neural spike observations rather than BOLD; adopt its direction-of-prediction reasoning, not its Poisson likelihood blindly. BrainIAK introduces run labels and hemodynamic timing for real fMRI decoding. A real exercise requires an event table, image mask, alignment checks, and independently held-out runs.

**Primary teaching sources, pinned where hosted on GitHub:**

- [NMA: GLMs for Encoding](https://github.com/NeuromatchAcademy/course-content/blob/44634e960df7a14cd0bf7398187f2d209d26b0e8/tutorials/W1D3_GeneralizedLinearModels/student/W1D3_Tutorial1.ipynb)
- [BrainIAK: fMRI classification](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/03-classification.ipynb)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Is decoding proof of causal coding? **No:** prediction can use correlated consequences or nuisance signals.
2. Why can averaging erase a signal? **Opposite signed responses can sum to zero despite informative individual measurements.**

### Return to the research question

Revisit [PM01](../../curriculum/papers/modeling.md#pm01) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
